# Phase 7: publish the model on the Hugging Face Hub

Uploads the merged `mix_tag` checkpoint (`runs/finetune_rumble-2-flash_mix_tag/best` on Drive) with the model card
`docs/model_card.md` from the GitHub repo, then loads it back from the Hub in a fresh process and runs the exact
code from the model card on 3 test clips. The transcripts must match the saved test predictions
(`runs/rumble-2-flash_mix_tag_greedy_test/predictions.jsonl`). Any GPU works; on an L4 (bf16) the match should be
exact. You need a Hugging Face **write** token; you paste it into the login box yourself.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
CY = '/content/drive/MyDrive/cyasr'
BEST = f'{CY}/runs/finetune_rumble-2-flash_mix_tag/best'
HF_USER = 'manolisfosteris'
REPO_ID = f'{HF_USER}/rumble-2-flash-cypriot-greek'
CARD_URL = 'https://raw.githubusercontent.com/manolisfosteris/cypriot-greek/main/docs/model_card.md'

!pip install -q transformers==5.22.0 soundfile huggingface_hub
!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls -la {BEST}

In [ ]:
from huggingface_hub import notebook_login
notebook_login()   # paste a write token from https://huggingface.co/settings/tokens

In [ ]:
# Model card: fetched from GitHub, model id filled in
import requests, os
card = requests.get(CARD_URL, timeout=60).text.replace('MODEL_ID', REPO_ID)
assert 'MODEL_ID' not in card and card.startswith('---')
open('/content/README.md', 'w').write(card)
print(card[:400])

In [ ]:
# Upload (public). Only the model files in best/ plus the card.
from huggingface_hub import HfApi
api = HfApi()
print('logged in as', api.whoami()['name'])
api.create_repo(REPO_ID, repo_type='model', private=False, exist_ok=True)
api.upload_folder(folder_path=BEST, repo_id=REPO_ID, commit_message='Merged LoRA checkpoint (mix_tag)')
api.upload_file(path_or_fileobj='/content/README.md', path_in_repo='README.md', repo_id=REPO_ID,
                commit_message='Model card')
print([f for f in api.list_repo_files(REPO_ID)])
print('https://huggingface.co/' + REPO_ID)

In [ ]:
# Test from the Hub in a fresh process: the model card's code on 3 test clips vs the saved predictions
import json, subprocess
preds = [json.loads(l) for l in open(f'{CY}/runs/rumble-2-flash_mix_tag_greedy_test/predictions.jsonl')][:3]
names = [f'data/audios/spontaneous-speech-el-CY-{p["audio_id"]}.mp3' for p in preds]
!mkdir -p /content/clips && unzip -o -j -q {CY}/cyasr_bundle.zip {' '.join(names)} -d /content/clips
test = f'''
import json, torch, soundfile as sf
from scipy.signal import resample_poly
from transformers import AutoProcessor, Qwen3ASRForConditionalGeneration
model_id = "{REPO_ID}"
processor = AutoProcessor.from_pretrained(model_id)
model = Qwen3ASRForConditionalGeneration.from_pretrained(model_id, dtype=torch.bfloat16).to("cuda").eval()
for p in {json.dumps([{"id": p["audio_id"], "hyp": p["hyp"]} for p in preds], ensure_ascii=False)}:
    audio, sr = sf.read(f"/content/clips/spontaneous-speech-el-CY-{{p['id']}}.mp3", dtype="float32", always_2d=True)
    audio = audio.mean(axis=1)
    if sr != 16000:
        audio = resample_poly(audio, 16000 // __import__("math").gcd(sr, 16000), sr // __import__("math").gcd(sr, 16000)).astype("float32")
    inputs = processor.apply_transcription_request(
        [audio], language=["el"], prompt="Cypriot Greek", digits=False, filler=True, verbatim=True,
    ).to(model.device, model.dtype)
    out = model.generate(**inputs, max_new_tokens=4096, do_sample=False)
    hyp = processor.decode(out[:, inputs["input_ids"].shape[1]:], return_format="transcription_only")[0]
    print("MATCH" if hyp.strip() == p["hyp"].strip() else "DIFF", "|", hyp, "|", p["hyp"])
'''
open('/content/test_hub.py', 'w').write(test)
r = subprocess.run(['python', '/content/test_hub.py'], capture_output=True, text=True,
                   env={**os.environ, 'HF_HOME': '/content/hf_fresh'})
print(r.stdout[-3000:]); print(r.stderr[-2000:] if r.returncode else 'exit 0')